# 04 Contribution 4 — Temporal / concept-drift
Needs a timestamped dataset (AndroZoo/KronoDroid) with a `year` column.

In [7]:
import sys; sys.path.insert(0,'..')
from ream.contributions import c4_temporal as c4
from ream.core.kronodroid import load_kronodroid_with_year
import pandas as pd

In [8]:
import warnings
warnings.filterwarnings("ignore") 

In [9]:

# loads the real-device parquet, splits into the 3 modalities, parses the year
mods, y, year = load_kronodroid_with_year(device="real")   # or device="emu"

# run the temporal-vs-random comparison for each modality
for tag, X in mods.items():
    df = X.copy()
    df["label"] = y.values
    df["year"]  = year.values
    df = df.dropna(subset=["year"])          # drop samples with invalid/missing dates
    feat = [c for c in df.columns if c not in ("label", "year")]
    res = c4.run_temporal(df, feat, "label", "year",
                          cutoff=2016, model_kind="lgbm", seed=0)
    print(tag,
          "| stability drop under drift:", round(res["stability_drop"], 3),
          "| faithfulness change:",        round(res["faithfulness_change"], 3))

K1_Syscall | stability drop under drift: 0.054 | faithfulness change: -0.085
K2_Perm | stability drop under drift: -0.042 | faithfulness change: -0.144
K3_Static | stability drop under drift: 0.0 | faithfulness change: 0.014
